In [227]:
import json
import re
from collections import Counter
from augur.utils import json_to_tree
from Bio import SeqIO
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.gridspec import GridSpec
import pandas as pd
import numpy as np

## trying to implement what I've done manually with code
## (find muts that occur along the trunk just before egg muts start occurring or after they stop)
## so far this isn't workign

In [136]:
def get_full_trunk(virus):
    """
    get the full series of trunk muts
    """
    
    tree_path= f'../nextstrain_builds/egg-enriched/auspice/{virus}_ha_egg.json'
    
    #read in the tree
    with open(tree_path, 'r') as f:
        tree_json = json.load(f)
        
    #put tree in Bio.phylo format
    tree = json_to_tree(tree_json)
    
    # make dictionary mapper with HA1 muts per node
    node_to_ha1_muts = {}
    node_to_nuc_muts = {}
    
    for node in tree.find_clades():
        ha1_muts = node.branch_attrs['mutations'].get('HA1', [])
        nuc_muts = node.branch_attrs['mutations'].get('nuc', [])
        node_to_ha1_muts[node.name] = ha1_muts
        node_to_nuc_muts[node.name] = nuc_muts
    
    # find max date in tree
    max_date= 1990
    for node in tree.find_clades(terminal=True):
        if node.node_attrs['num_date']['value']>max_date:
            max_date = node.node_attrs['num_date']['value']
    
    # for all recent tips (within 3 months of max_date)
    # get path from root 
    recent_paths = []
    for node in tree.find_clades(terminal=True):
        if node.node_attrs['num_date']['value']>max_date-0.25:
            # path back to root
            path = tree.get_path(node)
            recent_paths.append([x.name for x in path])
            
    trunk_path = [x for x in recent_paths[0] if all(x in lst for lst in recent_paths[1:])]
    
    trunk_ha1_muts_by_node = {x:node_to_ha1_muts[x] for x in trunk_path}
    trunk_nuc_muts_by_node = {x:node_to_nuc_muts[x] for x in trunk_path}

    
    # get date for each trunk branch
    trunk_dates= {}
    for node in tree.find_clades():
        if node.name in trunk_path:
            trunk_dates[node.name] = node.node_attrs['num_date']['value']
    
    
    return trunk_path, trunk_dates, trunk_ha1_muts_by_node, trunk_nuc_muts_by_node

In [38]:
def read_in_adaptive_muts(virus):
    """
    Read in the adaptive mutations
    """
    
    # read in all adative sites
    adaptive_mut_file = f"../egg-mutation-analysis/egg-adaptive-muts/ha_HA1_adaptive-muts.json"
    
    with open(adaptive_mut_file) as json_handle:
        egg_muts_info = json.load(json_handle)
        
    # get the HA1 adative muts for this virus
    ha1_adaptive_sites= egg_muts_info['all_adaptive'][virus]
        
    #get adpative aas at this res
    aas_at_adaptive_sites = egg_muts_info['aas_at_adaptive_sites'][virus]
    
    all_adaptive_subs = [f'{k}{x}' for k,v in aas_at_adaptive_sites.items() for x in v]
    
    return all_adaptive_subs

In [48]:
def get_strains_w_adaptive_muts(virus):
    """
    Get strains that have each egg adaptive mut
    """
    
    curated_mut_file = f"../egg-mutation-analysis/egg-muts-by-strain/{virus}_ha_curated-egg-muts.json"
    
    with open(curated_mut_file) as json_handle:
        egg_mut_info = json.load(json_handle)
    
    # all adaptive muts for this virus
    all_adaptive_subs = read_in_adaptive_muts(virus)
    
    # dict to store all strains that have the mut
    strains_by_adaptive_mut = {x: [] for x in all_adaptive_subs}
    
    for strain, muts in egg_mut_info.items():
        for m in muts['HA1']:
            # check if this is an adaptive mut
            if m in all_adaptive_subs:
                # if so, add this strain to the list of all strains with this mut
                strains_by_adaptive_mut[m].append(strain)
    
    return strains_by_adaptive_mut

In [194]:
def get_date_limits_for_mut(virus, mut):
    """
    For a given egg mut, find the date range it occurs over
    """
    # get all egg strains with this mut
    strains_by_adaptive_mut = get_strains_w_adaptive_muts(virus)
    # if there are multiple AAs that should be grouped together
    if '|' in mut:
        split_between_aa = mut.split('|')
        codon= split_between_aa[0][:-1]
        aas = [split_between_aa[0][-1]]+split_between_aa[1:]
        strains_w_this_mut = []
        for aa in aas:
            strains_w_this_aa_mut = strains_by_adaptive_mut[f'{codon}{aa}']
            strains_w_this_mut+=strains_w_this_aa_mut
    else:
        strains_w_this_mut = strains_by_adaptive_mut[mut]
    
    # read in tree
    tree_path= f'../nextstrain_builds/egg-enriched/auspice/{virus}_ha_egg.json'
    
    #read in the tree
    with open(tree_path, 'r') as f:
        tree_json = json.load(f)
        
    #put tree in Bio.phylo format
    tree = json_to_tree(tree_json)
    
    min_date_mut = 2025
    max_date_mut = 1990
    
    min_date_overall = 2025
    max_date_overall = 1990
    # find date range
    for node in tree.find_clades(terminal=True):
        date = node.node_attrs['num_date']['value']
        if date > max_date_overall:
            max_date_overall = date
        if date < min_date_overall:
            min_date_overall = date
            
        if node.name in strains_w_this_mut:
            if date > max_date_mut:
                max_date_mut = date
            if date < min_date_mut:
                min_date_mut = date
    
    # see if the mut was present early on and then stopped occurring
    # or whether it wasn't initially present and then started occurring
    # or both
    present_initially= False
    present_at_end = False
    if min_date_mut - min_date_overall <= 2:
        present_initially = True
    if max_date_overall - max_date_mut <= 2:
        present_at_end = True
        
        
    return present_initially, present_at_end

In [215]:
def get_trunk_section_for_mut(virus, mut):
    """
    For a given egg-mut, find the section of the trunk that strains with this mut descend from
    To allow for outliers, exclude trunk branches if only 5% or less of strains 
    have that certain portion of the trunk
    """
    
    strains_by_adaptive_mut = get_strains_w_adaptive_muts(virus)
    # if there are multiple AAs that should be grouped together
    if '|' in mut:
        split_between_aa = mut.split('|')
        codon= split_between_aa[0][:-1]
        aas = [split_between_aa[0][-1]]+split_between_aa[1:]
        strains_w_this_mut = []
        for aa in aas:
            strains_w_this_aa_mut = strains_by_adaptive_mut[f'{codon}{aa}']
            strains_w_this_mut+=strains_w_this_aa_mut
    else:
        strains_w_this_mut = strains_by_adaptive_mut[mut]
    
    # get tree trunk
    trunk_path, trunk_dates, trunk_ha1_muts_by_node, trunk_nuc_muts_by_node = get_full_trunk(virus)
    
    tree_path= f'../nextstrain_builds/egg-enriched/auspice/{virus}_ha_egg.json'
    
    #read in the tree
    with open(tree_path, 'r') as f:
        tree_json = json.load(f)
        
    #put tree in Bio.phylo format
    tree = json_to_tree(tree_json)
    
    # based on date, see whether this mut was present initially, or whether it started occurring later
    # and whether it stops occurring or not
    present_initially, present_at_end = get_date_limits_for_mut(virus, mut)
    
    # list of all trunk paths for strains with this mut
    # (meaning the portion of the trunk that the strain's path follows)
    paths_of_strains_w_mut = []
    # count the number of strains with this mut that traverse each branch in trunk path
    count_strains_w_branch_in_path = {x:0 for x in trunk_path}

    for node in tree.find_clades(terminal=True):
        if node.name in strains_w_this_mut:
            path = [x.name for x in tree.get_path(node) if x.name in trunk_path]
            paths_of_strains_w_mut.append(path)
            for p in path:
                count_strains_w_branch_in_path[p]+=1
        
        
    if present_at_end:
        # find the most recent trunk branch that is on the path of at least 95% of strains with this mut
        cutoff_num = len(strains_w_this_mut)*0.95
        basal_branch = ''
        for t in trunk_path:
            if count_strains_w_branch_in_path[t] >=cutoff_num:
                basal_branch = t
                
                
        node_where_muts_start = basal_branch
    else:
        node_where_muts_start = 'no_start'
        

    if present_initially:
        # find where muts stop
#         cutoff_num = len(strains_w_this_mut)*0.05

        for t in trunk_path:
            if count_strains_w_branch_in_path[t] >1:
                node_where_muts_end = t
        
    else:
        node_where_muts_end = 'no_end'
        

    return node_where_muts_start, node_where_muts_end
            

In [178]:
def determine_codon(virus, mut):
    """
    Find what codon mut is in
    """
    

    reference_file = f'../nextstrain_builds/egg-enriched/config/{virus}/ha/genemap.gff'
    
    
    with open(reference_file, 'r') as gff_handle:
        gff_lines = gff_handle.readlines()
        for line in gff_lines:
            if 'gene' in line.split('\t'):
                if 'HA1' in line.split('\t')[-1]:
                    ha1_location = [int(line.split('\t')[3]), int(line.split('\t')[4])]

    # split nt positions into codons                
    ha1_codons =[[i, i+1, i+2] for i in range(ha1_location[0], ha1_location[1] + 1, 3)]
    # map each nt position to what codon it is in (1 based)
    nuc_pos_to_codon = {x:i+1 for i,c in enumerate(ha1_codons) for x in c}

    # see if this mut is in HA1
    nuc_mut_pos = int(mut[1:-1])
    
    if nuc_mut_pos in range(ha1_location[0], ha1_location[1]):
        ha1_codon_this_mut = nuc_pos_to_codon[nuc_mut_pos]
        
    else:
        ha1_codon_this_mut = 'notHA1'

                    
    return ha1_codon_this_mut

In [268]:
def get_potential_background_dependence(virus, mut):
    """
    Look within 0.5 year of the branch identified as the stopping or ending point of the mutation
    Get muts on all trunk branches to get a list of potenital background changes that influence mut
    Priority:
    1. other mutation in that codon
    2. nearby mutation
    3. manual inspection
    """
    
    # get trunk branches where egg muts start or end
    node_where_muts_start, node_where_muts_end = get_trunk_section_for_mut(virus, mut)
    
    if node_where_muts_start == 'no_start' and node_where_muts_end == 'no_end':
        dependence = False
    else:
        dependence = True
        
        
    # get tree trunk
    trunk_path, trunk_dates, trunk_ha1_muts_by_node, trunk_nuc_muts_by_node = get_full_trunk(virus)
    
    if '|' in mut:
        this_codon = int(mut.split('|')[0][:-1])
    else:
        this_codon = int(mut[:-1])
    
    mut_in_same_codon_start = []
    other_muts_in_HA1_start = []
    # get branches within 0.5year of  muts starting or ending (allow either side for incomplete fixations)
    if node_where_muts_start!= 'no_start':
        start_date = trunk_dates[node_where_muts_start]
        branches_near_start = [x for x,d in trunk_dates.items() if abs(start_date-d)<=0.5]
        
        ha1_muts_on_branches_near_start = [trunk_ha1_muts_by_node[x] for x in branches_near_start]
        nuc_muts_on_branches_near_start = [trunk_nuc_muts_by_node[x] for x in branches_near_start]
        # flatten these lists 
        ha1_muts_on_branches_near_start = [item for sublist in ha1_muts_on_branches_near_start for item in sublist]
        nuc_muts_on_branches_near_start = [item for sublist in nuc_muts_on_branches_near_start for item in sublist]
        
        # see if any are in same codon as this mut
        for nt_mut in nuc_muts_on_branches_near_start:
            ha1_codon_this_mut = determine_codon(virus, nt_mut)
            
            if ha1_codon_this_mut == this_codon:
                mut_in_same_codon_start.append(nt_mut)
                
        # see what other aa subs there are in HA1
        other_muts_in_HA1_start = ha1_muts_on_branches_near_start
        
    
    mut_in_same_codon_end = []
    other_muts_in_HA1_end = []
    
    if node_where_muts_end!='no_end':
        end_date = trunk_dates[node_where_muts_end]
        branches_near_end = [x for x,d in trunk_dates.items() if abs(end_date-d)<=1.0]
        
        ha1_muts_on_branches_near_end = [trunk_ha1_muts_by_node[x] for x in branches_near_end]
        nuc_muts_on_branches_near_end = [trunk_nuc_muts_by_node[x] for x in branches_near_end]
        # flatten these lists 
        ha1_muts_on_branches_near_end = [item for sublist in ha1_muts_on_branches_near_end for item in sublist]
        nuc_muts_on_branches_near_end = [item for sublist in nuc_muts_on_branches_near_end for item in sublist]
        
        # see if any are in same codon as this mut
        for nt_mut in nuc_muts_on_branches_near_end:
            ha1_codon_this_mut = determine_codon(virus, nt_mut)
            
            if ha1_codon_this_mut == this_codon:
                mut_in_same_codon_end.append(nt_mut)
                
        # see what other aa subs there are in HA1
        other_muts_in_HA1_end = ha1_muts_on_branches_near_end
        
    
    return dependence, mut_in_same_codon_start, other_muts_in_HA1_start, mut_in_same_codon_end, other_muts_in_HA1_end

In [242]:
def get_genotype_at_X(virus, egg_mut, pos):
    """
    For all strains with this egg_mut, find the genotype at the other posiiotn
    """
    
    # get all egg strains with this mut
    strains_by_adaptive_mut = get_strains_w_adaptive_muts(virus)
    # if there are multiple AAs that should be grouped together
    if '|' in egg_mut:
        split_between_aa = egg_mut.split('|')
        codon= split_between_aa[0][:-1]
        aas = [split_between_aa[0][-1]]+split_between_aa[1:]
        strains_w_this_mut = []
        for aa in aas:
            strains_w_this_aa_mut = strains_by_adaptive_mut[f'{codon}{aa}']
            strains_w_this_mut+=strains_w_this_aa_mut
    else:
        strains_w_this_mut = strains_by_adaptive_mut[egg_mut]
    
    # store all genotypes at pos (for strains with egg mut)
    genotype_at_pos = []
    
    # read in translated HA1 for all tips
    translations_path = f'../nextstrain_builds/egg-enriched/results/{virus}/ha/translations/HA1.fasta'
    
    for record in SeqIO.parse(translations_path, 'fasta'):
        strain = record.id
        
        if strain in strains_w_this_mut:
            # convert to 1-based
            if record.seq[pos-1] not in ['X', '-']:
                genotype_at_pos.append(record.seq[pos-1])
            
    count_genotype_at_pos= Counter(genotype_at_pos)
    
    pct_w_genotype_at_pos = {a:c/sum(count_genotype_at_pos.values()) for a,c in count_genotype_at_pos.items()}
    
    return pct_w_genotype_at_pos

In [271]:
def narrow_down_potentials(virus, mut):
    """
    From the potential trunk muts causing background dependence, 
    narrow down the candidates by looking for percent of strains with egg mut 
    that have the background mut (or lack of it)
    """
    
    (dependence, mut_in_same_codon_start, other_muts_in_HA1_start, 
     mut_in_same_codon_end, other_muts_in_HA1_end) = get_potential_background_dependence(virus, mut)
    if dependence==False:
        print('no background dependence')
    
    elif dependence==True:
        if len(mut_in_same_codon_start)!=0:
            print(f'mut happens after {mut_in_same_codon_start} in same codon')
        if len(mut_in_same_codon_end)!=0:
            print(f'mut stops after {mut_in_same_codon_end} in same codon')
            
        if len(mut_in_same_codon_start)==0 and len(mut_in_same_codon_end)==0:
            
            if len(other_muts_in_HA1_start)!=0:
                # percentage with the candidate mut
                percent_w_mut_by_candidate = {}

                for m in other_muts_in_HA1_start:
                    m_pos = int(m[1:-1])

                    pct_w_genotype_at_pos = get_genotype_at_X(virus, mut, m_pos)
                    pct_w_mut = pct_w_genotype_at_pos[m[-1]]
                    percent_w_mut_by_candidate[m] = pct_w_mut

                print(f'background that happen around when this egg mut appears: {other_muts_in_HA1_start}. \nOf these, the following percent of strains with egg mut have that background mut: {percent_w_mut_by_candidate}')



            if len(other_muts_in_HA1_end)!=0:
                # percentage without the candidate mut
                percent_wo_mut_by_candidate = {}

                for m in other_muts_in_HA1_end:
                    m_pos = int(m[1:-1])

                    pct_w_genotype_at_pos = get_genotype_at_X(virus, mut, m_pos)

                    pct_wo_mut = pct_w_genotype_at_pos[m[0]]
                    percent_wo_mut_by_candidate[m] = pct_wo_mut

                print(f'background that happen around when this egg mut appears: {other_muts_in_HA1_end}. \nOf these, the following percent of strains with egg mut have that background mut: {percent_wo_mut_by_candidate}')




        
        

In [257]:
narrow_down_potentials('h1n1pdm', '187V|N|T')

background that happen around when this egg mut appears: ['S164T', 'S183P']. 
Of these, the folloing percent of strains with egg mut have that background mut: {'S164T': 0.9916666666666667, 'S183P': 1.0}


In [259]:
narrow_down_potentials('h1n1pdm', '191I')

background that happen around when this egg mut appears: []. 
Of these, the following percent of strains with egg mut have that background mut: {'S74R': 0.9761904761904762, 'I295V': 0.9523809523809523, 'S164T': 0.9879518072289156, 'S183P': 0.9759036144578314, 'N260D': 1.0}


In [260]:
narrow_down_potentials('h1n1pdm', '127E')

background that happen around when this egg mut appears: []. 
Of these, the following percent of strains with egg mut have that background mut: {'N129D': 0.9104477611940298, 'T185I': 0.746268656716418, 'N156K': 0.9850746268656716, 'K130N': 1.0, 'L161I': 0.9850746268656716}


In [261]:
narrow_down_potentials('h1n1pdm', '222N|G')

background that happen around when this egg mut appears: ['S203T', 'S185T', 'D97N']. 
Of these, the following percent of strains with egg mut have that background mut: {'S203T': 0.9720670391061452, 'S185T': 0.553072625698324, 'D97N': 0.7318435754189944}
background that happen around when this egg mut appears: ['S203T', 'S185T', 'D97N']. 
Of these, the following percent of strains with egg mut have that background mut: {'V250A': 0.8491620111731844, 'A186T': 0.8324022346368715, 'Q189E': 0.8435754189944135, 'E224A': 0.8547486033519553, 'K54Q': 0.8547486033519553, 'R259K': 0.8547486033519553, 'K308R': 0.8547486033519553}


In [264]:
narrow_down_potentials('h1n1pdm', '223R')

background that happen around when this egg mut appears: ['S203T', 'S185T', 'D97N']. 
Of these, the following percent of strains with egg mut have that background mut: {'S203T': 0.9787234042553191, 'S185T': 0.5484633569739953, 'D97N': 0.8416075650118203}
background that happen around when this egg mut appears: ['S203T', 'S185T', 'D97N']. 
Of these, the following percent of strains with egg mut have that background mut: {'V250A': 0.735224586288416, 'A186T': 0.7659574468085106, 'Q189E': 0.7683215130023641, 'E224A': 0.7706855791962175, 'K54Q': 0.7659574468085106, 'R259K': 0.7706855791962175, 'K308R': 0.7730496453900709}


In [269]:
narrow_down_potentials('h3n2', '193R')

mut stops after ['A654T', 'G655T'] in same codon


In [273]:
narrow_down_potentials('h3n2', '194P')

no background dependence
